# 04 — Improved Model (v2)
**Baseline score: 0.7078  →  Target: 0.75+**

Improvements over v1:
1. **Threshold optimisation** on OOF predictions (F1 was 0.648 at threshold=0.5)
2. **New features**: temporal trend (slope), coefficient of variation, interaction features
3. **CatBoost** added to the ensemble (optional — install with `uv add catboost`)

Key insight: `TargetF1` (binary) and `TargetRAUC` (proba) are scored independently.
The threshold only affects F1 — AUC is unaffected. Finding the optimal threshold on OOF
before submission is the cheapest possible gain.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.features import prepare_Xy
from src.train import cross_validate, ensemble_predict, blend, save_models
from src.evaluate import find_best_threshold, print_scores, zindi_score
from src.predict import make_submission, load_threshold

DATA = pathlib.Path('..') / 'data' / 'raw'
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

## 1. Build features (v2 — includes trend, CV, interaction)

In [ ]:
X_train, y_train, X_test = prepare_Xy(DATA / 'Train.csv', DATA / 'Test.csv')
print(f'X_train: {X_train.shape}  |  X_test: {X_test.shape}')

# Show new feature names
new_feats = [c for c in X_train.columns if any(x in c for x in ['trend', '_cv', 'water_score', 'pond_signal', 'vh_ndwi'])]
print(f'\nNew features added ({len(new_feats)}):', new_feats)

## 2. New feature distributions by class

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
axes = axes.flatten()

key_new = ['ndwi_trend', 'ndvi_trend', 'ndwi_cv', 'water_score', 'pond_signal', 'vh_ndwi']
for ax, col in zip(axes, key_new):
    for label, color, name in [(0, '#4C72B0', 'No pond'), (1, '#DD8452', 'Pond')]:
        vals = X_train.loc[y_train == label, col].dropna()
        ax.hist(vals, bins=40, alpha=0.65, color=color, label=name, density=True)
    ax.set_title(col)
    ax.legend(fontsize=8)

plt.suptitle('New v2 feature distributions by class', fontsize=13)
plt.tight_layout(); plt.show()

## 3. LightGBM — with optimal threshold

In [ ]:
print('=== LightGBM ===')
lgb_oof, lgb_models, lgb_thr = cross_validate(X_train, y_train, model_type='lgb')

## 4. XGBoost — with optimal threshold

In [ ]:
print('=== XGBoost ===')
xgb_oof, xgb_models, xgb_thr = cross_validate(X_train, y_train, model_type='xgb')

## 5. CatBoost (optional)

In [ ]:
try:
    import catboost
    print('=== CatBoost ===')
    cb_oof, cb_models, cb_thr = cross_validate(X_train, y_train, model_type='cb')
    HAS_CB = True
except ImportError:
    print('CatBoost not installed. Run: uv add catboost')
    HAS_CB = False
    cb_oof = None

## 6. Ensemble blend + optimal threshold

In [ ]:
if HAS_CB and cb_oof is not None:
    blended_oof = blend([lgb_oof, xgb_oof, cb_oof], weights=[0.5, 0.3, 0.2])
    blend_tag = 'lgb_xgb_cb'
else:
    blended_oof = blend([lgb_oof, xgb_oof], weights=[0.6, 0.4])
    blend_tag = 'lgb_xgb'

best_thr, best_f1 = find_best_threshold(y_train.values, blended_oof)

print(f'\n--- Blended OOF at threshold=0.50 ---')
print_scores(y_train.values, blended_oof, threshold=0.50)
print(f'\n--- Blended OOF at optimal threshold={best_thr:.3f} ---')
print_scores(y_train.values, blended_oof, threshold=best_thr)

In [ ]:
# Threshold sweep — visualise F1 and Zindi score vs threshold
thresholds = np.linspace(0.05, 0.95, 150)
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score

auc_val  = roc_auc_score(y_train.values, blended_oof)
f1_vals  = [f1_score(y_train.values, (blended_oof >= t).astype(int), zero_division=0) for t in thresholds]
zindi_vals = [0.6 * f1 + 0.4 * auc_val for f1 in f1_vals]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(thresholds, f1_vals,    label='F1', color='steelblue')
ax.plot(thresholds, zindi_vals, label='Zindi (0.6×F1 + 0.4×AUC)', color='coral')
ax.axvline(best_thr, color='green', linestyle='--', label=f'Optimal={best_thr:.3f}')
ax.axvline(0.5,      color='gray',  linestyle=':',  label='Default=0.5')
ax.set_xlabel('Threshold')
ax.set_title('Threshold sweep on blended OOF predictions')
ax.legend()
plt.tight_layout(); plt.show()
print(f'F1 gain from threshold optimisation: {best_f1 - f1_vals[75]:.4f}')

## 7. Feature importance (v2)

In [ ]:
importances = np.stack([m.booster_.feature_importance(importance_type='gain') for m in lgb_models])
feat_imp = pd.Series(importances.mean(axis=0), index=lgb_models[0].feature_name_).sort_values(ascending=False)

# Highlight new features
top30 = feat_imp.head(30).sort_values()
colors = ['#DD8452' if any(x in c for x in ['trend','_cv','water_score','pond_signal','vh_ndwi']) 
          else '#4C72B0' for c in top30.index]

fig, ax = plt.subplots(figsize=(9, 8))
top30.plot.barh(ax=ax, color=colors)
ax.set_title('Top-30 LGB feature importance — orange = new v2 features')
plt.tight_layout(); plt.show()

## 8. Save models and generate submission

In [ ]:
import json
from pathlib import Path

save_models(lgb_models, 'lgb')
save_models(xgb_models, 'xgb')
if HAS_CB:
    save_models(cb_models, 'cb')

MODEL_DIR = pathlib.Path('..') / 'models'
MODEL_DIR.mkdir(exist_ok=True)
thr_path = MODEL_DIR / 'thresholds.json'
thresholds_dict = {'blend': best_thr, 'lgb': lgb_thr, 'xgb': xgb_thr}
if HAS_CB:
    thresholds_dict['cb'] = cb_thr
with open(thr_path, 'w') as f:
    json.dump(thresholds_dict, f, indent=2)
print('Thresholds:', thresholds_dict)

In [ ]:
from src.predict import predict_proba as ensemble_pred_proba

lgb_test  = ensemble_pred_proba(lgb_models, X_test)
xgb_test  = ensemble_pred_proba(xgb_models, X_test)

if HAS_CB:
    cb_test  = ensemble_pred_proba(cb_models, X_test)
    test_proba = blend([lgb_test, xgb_test, cb_test], weights=[0.5, 0.3, 0.2])
else:
    test_proba = blend([lgb_test, xgb_test], weights=[0.6, 0.4])

sub = make_submission(X_test.index, test_proba, threshold=best_thr, tag=f'v2_{blend_tag}_thr{best_thr:.3f}')
print(f"\nPositives predicted: {sub['TargetF1'].sum()} / {len(sub)}")
sub.head()